# Calendar Dimension

Builds the standard calendar table `silver.calendar` with one row per day from 1940-01-01 to 2030-12-31.

In [0]:
from pyspark.sql import functions as F

START_DATE = "1940-01-01"
END_DATE = "2030-12-31"

# One row per day in the range
dates_df = spark.sql(
    f"SELECT explode(sequence(DATE'{START_DATE}', DATE'{END_DATE}', INTERVAL 1 DAY)) AS date"
)

d = F.col("date")

calendar_df = (
    dates_df
    .select(
        F.date_format(d, "yyyyMMdd").cast("int").alias("datekey"),
        d.alias("date"),
        # Year
        F.year(d).alias("year"),
        # Quarter
        F.quarter(d).alias("quarter"),
        F.concat(F.lit("Q"), F.quarter(d)).alias("quarter_name"),
        F.concat(F.year(d), F.lit("-Q"), F.quarter(d)).alias("year_quarter"),
        # Month
        F.month(d).alias("month"),
        F.date_format(d, "MMMM").alias("month_name"),
        F.date_format(d, "MMM").alias("month_short_name"),
        F.date_format(d, "yyyy-MM").alias("year_month"),
        (F.year(d) * 100 + F.month(d)).alias("year_month_key"),
        # ISO week
        F.weekofyear(d).alias("week_of_year"),
        F.expr("extract(YEAROFWEEK FROM date)").alias("iso_year"),
        F.ceil(F.dayofmonth(d) / 7).cast("int").alias("week_of_month"),
        F.date_trunc("week", d).cast("date").alias("week_start_date"),
        F.date_add(F.date_trunc("week", d).cast("date"), 6).alias("week_end_date"),
        # Day
        F.dayofmonth(d).alias("day"),
        F.dayofyear(d).alias("day_of_year"),
        F.expr("extract(DAYOFWEEK_ISO FROM date)").alias("day_of_week"),  # 1=Mon .. 7=Sun
        F.date_format(d, "EEEE").alias("day_name"),
        F.date_format(d, "EEE").alias("day_short_name"),
        (F.datediff(d, F.date_trunc("quarter", d).cast("date")) + 1).alias("day_of_quarter"),
        # Flags
        F.expr("extract(DAYOFWEEK_ISO FROM date)").isin(6, 7).alias("is_weekend"),
        (~F.expr("extract(DAYOFWEEK_ISO FROM date)").isin(6, 7)).alias("is_weekday"),
        (d == F.trunc(d, "month")).alias("is_month_start"),
        (d == F.last_day(d)).alias("is_month_end"),
        (d == F.date_trunc("quarter", d).cast("date")).alias("is_quarter_start"),
        (d == F.date_sub(F.add_months(F.date_trunc("quarter", d).cast("date"), 3), 1)).alias("is_quarter_end"),
        ((F.month(d) == 1) & (F.dayofmonth(d) == 1)).alias("is_year_start"),
        ((F.month(d) == 12) & (F.dayofmonth(d) == 31)).alias("is_year_end"),
        F.expr("(year(date) % 4 = 0 AND year(date) % 100 != 0) OR year(date) % 400 = 0").alias("is_leap_year"),
        # Period boundaries
        F.trunc(d, "month").alias("month_start_date"),
        F.last_day(d).alias("month_end_date"),
        F.date_trunc("quarter", d).cast("date").alias("quarter_start_date"),
        F.date_sub(F.add_months(F.date_trunc("quarter", d).cast("date"), 3), 1).alias("quarter_end_date"),
        F.trunc(d, "year").alias("year_start_date"),
        F.dayofmonth(F.last_day(d)).alias("days_in_month"),
    )
)

display(calendar_df.limit(10))

In [0]:
(
    calendar_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.calendar")
)

In [0]:
%sql
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT datekey) AS distinct_datekeys,
  MIN(date) AS min_date,
  MAX(date) AS max_date,
  DATEDIFF(MAX(date), MIN(date)) + 1 AS expected_days,
  SUM(CASE WHEN is_leap_year AND month = 2 AND day = 29 THEN 1 ELSE 0 END) AS leap_days
FROM silver.calendar